In [1]:
import os
import json
from PyPDF2 import PdfReader
import ollama

In [2]:
OUTPUT_FILE = "train_data.jsonl"

In [3]:
def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    return "\n".join([page.extract_text() for page in reader.pages if page.extract_text()])


In [4]:
def generate_example(study_text):
    prompt = f"""
You are an expert in creating university-level examination question papers.

Your task is to create a well-structured question paper **based only on the given study material below**.
- Include sections like Part A (short answer), Part B (long answer), and Part C (essay).
- Cover important points from the material.
- Do not copy sentences directly from the text.
- Ensure the format is academic and resembles a university question paper.

### Study Material:
{study_text}

### Question Paper:
"""
    try:
        response = ollama.chat(model="mistral", messages=[{"role": "user", "content": prompt}])
        print("🔍 Raw Ollama Response:", response)
        return response.get("message", {}).get("content", "").strip()
    except Exception as e:
        print(f"❌ Error during generation: {e}")
        return ""


In [5]:
def add_pair_to_dataset(study_pdf_path):
    try:
        study_text = extract_text_from_pdf(study_pdf_path)

        print(f"✅ Processing:\nStudy Material: {study_pdf_path}")
        generated_output = generate_example(study_text)

        entry = {
            "prompt": f"Study Material:\n{study_text}",
            "response": generated_output
        }

        with open(OUTPUT_FILE, "a", encoding="utf-8") as f:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

        print(f"\n✅ Example successfully added to {OUTPUT_FILE}\n")

    except Exception as e:
        print(f"❌ Error: {e}")


In [6]:
if __name__ == "__main__":
    study_pdf_path = input("Enter path to Study Material PDF: ").strip()

    if not os.path.exists(study_pdf_path):
        print("❌ Study Material PDF not found.")
    else:
        add_pair_to_dataset(study_pdf_path)



Enter path to Study Material PDF:  /home/anjana/Downloads/c_study_1.pdf


✅ Processing:
Study Material: /home/anjana/Downloads/c_study_1.pdf
🔍 Raw Ollama Response: model='mistral' created_at='2025-04-18T13:41:08.252608326Z' done=True done_reason='stop' total_duration=614553484173 load_duration=14594302891 prompt_eval_count=2048 prompt_eval_duration=516213372436 eval_count=194 eval_duration=83678004508 message=Message(role='assistant', content=' This text appears to be part of a programming course material. It provides information about data types in C, which includes signed and unsigned integers, floating point numbers, characters, and void types. The table shows the size (in bits) and range for each type on a 16-bit machine.\n\nIt also explains that variable declaration tells the compiler the variable name and its data type, and must be done before using it in the program. The syntax for declaring a variable is provided. This material seems to be part of a section about declaring variables in C.\n\nAdditionally, there are some exercises at the end of this s